In [17]:
import numpy as np
import pandas as pd

In [18]:
df = pd.read_csv("evaluation/results.csv")

In [19]:
df

,run,evaluated_at,note,model_type,chain,step,model,checkpoint,epoch,batch_percent,...,wer_common_voice,wer_fleurs,wer_ukr_dialects,wer_cv10_test,cer_combined_weighted,cer_combined_mean,cer_common_voice,cer_fleurs,cer_ukr_dialects,cer_cv10_test
0,baseline/common-voice-only-baseline,2026-10-26 21:12:35,Бейзлан на CV 27.0 (витік dev у train),fine-tune,NaN,NaN,model-whisper-uk,model-whisper-uk-epoch00.ckpt,0.0,NaN,...,17.3980,11.5921,69.7700,15.6544,12.7225,12.3417,4.3564,3.6903,37.7449,3.5752
1,baseline/original-whisper-medium,2026-10-26 22:04:51,Evaluation оригінального Whisper Medium для по...,pretrained,NaN,NaN,openai/whisper-medium,NaN,NaN,NaN,...,20.9178,11.5430,73.6575,18.2911,15.9395,15.2404,5.9126,3.7007,46.6630,4.6854
2,baseline/whisper-uk-epoch00-01,2026-10-27 09:11:15,NaN,fine-tune,NaN,NaN,model-whisper-uk,model-whisper-uk-epoch00.ckpt,0.0,NaN,...,17.6380,11.3539,74.8863,15.3133,13.8852,13.4022,4.3707,3.3326,42.4985,3.4072
3,baseline/whisper-uk-epoch01-01,2026-10-27 09:48:35,NaN,fine-tune,NaN,NaN,model-whisper-uk,model-whisper-uk-epoch01.ckpt,1.0,NaN,...,17.5403,11.3399,73.8718,15.2302,13.8910,13.4188,4.3454,3.3721,42.5580,3.3995
4,student_whisper-commonvoice-based-mark5-epoch0...,2026-10-28 16:44:28,"Студент, на псевдах з найвищою оцінкою (5) від...",student,NaN,NaN,student_whisper-commonvoice-based,student_whisper-commonvoice-based-epoch01.ckpt,1.0,NaN,...,17.5555,11.6831,61.6888,15.7287,12.1387,11.7741,4.3817,3.7454,35.3970,3.5724
5,student_whisper-commonvoice-based-epoch01-01,2026-11-04 19:14:43,"Студент, навчений на всіх псевдах від VOA (оці...",student,NaN,NaN,student_whisper-commonvoice-based,student_whisper-commonvoice-based-epoch01.ckpt,1.0,NaN,...,18.0189,11.7532,74.2457,15.9124,13.7753,13.3531,4.4868,3.7662,41.5619,3.5975
6,student_whisper-commonvoice-based-epoch01-02,2026-11-06 08:30:06,NaN,student,NaN,NaN,student_whisper-commonvoice-based,student_whisper-commonvoice-based-epoch01.ckpt,1.0,NaN,...,17.7002,11.9353,65.1915,15.9648,12.5247,NaN,4.3739,3.8296,36.8645,3.6080
7,student_mix-m2345-cv-m45-cv-s1-pseudo-m2345-co...,2026-11-07 21:04:22,student_mix-m2345+cv-m45+cv step 1/2: pseudo-l...,student,student_mix-m2345+cv-m45+cv,1.0,student_mix-m2345+cv-m45+cv-s1-pseudo-m2345-co...,student_mix-m2345+cv-m45+cv-s1-pseudo-m2345-co...,0.0,NaN,...,15.4972,11.6131,70.0017,12.7465,13.2838,12.9666,3.9353,4.0459,41.0240,2.8611
8,student_mix-m2345-cv-m45-cv-s2-pseudo-m45-comm...,2026-11-08 02:21:29,student_mix-m2345+cv-m45+cv step 2/2: pseudo-l...,student,student_mix-m2345+cv-m45+cv,2.0,student_mix-m2345+cv-m45+cv-s2-pseudo-m45-comm...,student_mix-m2345+cv-m45+cv-s2-pseudo-m45-comm...,0.0,NaN,...,14.5894,11.6061,71.0425,11.6927,13.3869,13.1206,3.7519,4.2300,41.7991,2.7015
9,student_mix-m345-cv-m5-cv-s1-pseudo-m345-commo...,2026-11-08 15:26:31,student_mix-m345+cv-m5+cv step 1/2: pseudo-lab...,student,student_mix-m345+cv-m5+cv,1.0,student_mix-m345+cv-m5+cv-s1-pseudo-m345-commo...,student_mix-m345+cv-m5+cv-s1-pseudo-m345-commo...,0.0,NaN,...,16.7733,11.1438,62.2420,14.4475,12.0104,11.6243,4.1449,3.5115,35.6957,3.1450


In [20]:
# Актуальні результати (docs/thesis/thesis_plan.md, розд. 1 і 4). Показуються лише ці запуски.
# combined_weighted з results.csv включає CV10; «Разом» без CV10 —
# з docs/thesis/figures/combined_without_cv10.csv (не для всіх запусків).
KEEP_RUNS = [
    "baseline/original-whisper-medium",
    #"baseline/common-voice-only-baseline",
    #"student_mix-m345-cv-m5-cv-s1-pseudo-m345-common_voice-epoch00-01",
    #"student_mix-m345-cv-m5-cv-s2-pseudo-m5-common_voice-epoch00-01",
    #"student_mix-m2345-cv-m45-cv-s1-pseudo-m2345-common_voice-epoch00-01",
    #"student_mix-m2345-cv-m45-cv-s2-pseudo-m45-common_voice-epoch00-01",
    #"whisper-uk-cv-fixed-epoch01-01",
    "whisper-m-fixed-leaks-5-epochs-epoch04-01",
    "student_whisper-fixed-leaks-7-epochs-marks5-epoch06-01",
    "student_whisper-fixed-leaks-7-epochs-epoch06-01",
]
TEST_SETS = ["common_voice", "fleurs", "ukr_dialects"]

combined = pd.read_csv("docs/thesis/figures/combined_without_cv10.csv", usecols=["run", "wer_combined_no_cv10", "cer_combined_no_cv10"])
metrics = [f"{metric}_{test}" for metric in ("wer", "cer") for test in ["combined_weighted", *TEST_SETS]]
latest = (
    df.set_index("run")
    .reindex(KEEP_RUNS)[metrics]
    .join(combined.set_index("run"))
)
latest

,wer_combined_weighted,wer_common_voice,wer_fleurs,wer_ukr_dialects,cer_combined_weighted,cer_common_voice,cer_fleurs,cer_ukr_dialects,wer_combined_no_cv10,cer_combined_no_cv10
run,,,,,,,,,,
baseline/original-whisper-medium,34.6410,20.9178,11.5430,73.6575,15.9395,5.9126,3.7007,46.6630,37.3354,17.9530
whisper-m-fixed-leaks-5-epochs-epoch04-01,26.2772,10.6798,12.9789,66.7505,12.0751,2.8397,5.1273,38.6166,29.3963,13.9181
student_whisper-fixed-leaks-7-epochs-marks5-epoch06-01,29.1676,16.1524,13.3992,63.9999,12.8242,4.0778,5.9821,37.9395,NaN,NaN
student_whisper-fixed-leaks-7-epochs-epoch06-01,27.1576,14.2554,13.8334,61.3718,11.8329,3.7488,6.0819,35.0524,NaN,NaN
